# Random Forest Prediction of EPVI Indicators

This notebook trains one random forest model per EPVI indicator using:

- satellite-derived freguesia indicators;
- only the **basic** administrative indicators listed in `data/adm_data_split.json`.

The privately shared EPVI file is loaded from `pipeline/config/paths.*.json` via the `epvi_csv` key and is not stored in git. Detailed administrative indicators are intentionally excluded here; they are reserved for the later residual-explanation step.

When available, satellite predictors are loaded from the regenerated external pipeline output `outputs_indices_dir/freguesia_indices_streaming.csv`; otherwise the notebook falls back to the committed modeling snapshot `data/all_used_sat_indicators.csv`.


In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, ParameterGrid, cross_val_predict
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42

In [ ]:
# Resolve repository root and load local path config.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pipeline" / "config").exists()
)
sys.path.append(str(REPO_ROOT))

from importlib import import_module
from pipeline.utils.paths import load_paths, path_value, repo_data_path

rf_utils = import_module("pipeline.3_epvi_prediction.utils")
choose_satellite_csv = rf_utils.choose_satellite_csv
load_prediction_inputs = rf_utils.load_prediction_inputs
build_modeling_table = rf_utils.build_modeling_table
id_coverage_summary = rf_utils.id_coverage_summary
rmse = rf_utils.rmse
spearman_corr = rf_utils.spearman_corr

PATHS = load_paths()

SAT_CSV, REGENERATED_SAT_CSV, SNAPSHOT_SAT_CSV = choose_satellite_csv(PATHS, repo_data_path, path_value)
ADM_CSV = repo_data_path(PATHS, "all_used_adm_indicators.csv")
ADM_SPLIT_JSON = repo_data_path(PATHS, "adm_data_split.json")
EPVI_CSV = path_value(PATHS, "epvi_csv")

# Outputs stay outside git, next to the external data products.
MODEL_OUT_DIR = path_value(PATHS, "external_data_root") / "outputs" / "epvi_prediction" / "random_forest"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT)
print("satellite predictors:", SAT_CSV)
if SAT_CSV == SNAPSHOT_SAT_CSV:
    print("WARNING: using committed satellite snapshot; regenerated external index CSV was not found yet.")
print("administrative predictors:", ADM_CSV)
print("private EPVI targets:", EPVI_CSV)
print("model outputs:", MODEL_OUT_DIR)

In [ ]:
inputs = load_prediction_inputs(
    sat_csv=SAT_CSV,
    adm_csv=ADM_CSV,
    epvi_csv=EPVI_CSV,
    adm_split_json=ADM_SPLIT_JSON,
)

sat = inputs["sat"]
adm = inputs["adm"]
epvi = inputs["epvi"]
EPVI_TARGETS = inputs["targets"]
NAME_COL = inputs["name_col"]
sat_cols = inputs["sat_cols"]
basic_admin_cols = inputs["basic_admin_cols"]
detailed_admin_cols = inputs["detailed_admin_cols"]
predictor_cols = inputs["predictor_cols"]

print("Raw input table shapes:")
print("sat rows/cols:", sat.shape)
print("adm rows/cols:", adm.shape)
print("epvi rows/cols:", epvi.shape)
print()
print("Normalized ID coverage:")
display(id_coverage_summary(inputs))
print("targets:", EPVI_TARGETS)
print("name column:", NAME_COL)
print("candidate satellite predictors:", len(sat_cols))
print("candidate basic administrative predictors:", len(basic_admin_cols))
print("candidate predictors before merge/drop:", len(predictor_cols))
print("excluded detailed administrative predictors present:", len(detailed_admin_cols))


In [ ]:
# Merge modeling table.
model_df, predictor_cols, constant_cols = build_modeling_table(inputs)

missing_summary = (
    model_df[predictor_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
    .to_frame()
)

print("Final modeling rows after inner join:", len(model_df))
print("Final predictors used after dropping constant/all-missing columns:", len(predictor_cols))
print("Dropped constant/all-missing predictors:", constant_cols)
display(missing_summary.head(20))
display(model_df[["ID_norm", "parish_name", *EPVI_TARGETS]].head())


## Modeling Design

This notebook now runs a **focused second-stage** random-forest tuning step. The first broad search is recorded in `docs/model_tuning_history.md`; its completed targets consistently selected the same small part of the original search space.

The tuning search therefore evaluates the focused parameter grid with shuffled 3-fold CV and smaller forests for speed. The chosen configuration for each target is then evaluated with 5-fold out-of-fold predictions using an 800-tree forest. Detailed administrative predictors are still excluded; out-of-fold residuals remain the object for the residual-explanation stage.

In [ ]:
SEARCH_CV_FOLDS = 3
EVALUATION_CV_FOLDS = 5
SEARCH_N_ESTIMATORS = 400
FINAL_N_ESTIMATORS = 800

search_cv = KFold(n_splits=SEARCH_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
evaluation_cv = KFold(n_splits=EVALUATION_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), predictor_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

rf = RandomForestRegressor(
    n_estimators=SEARCH_N_ESTIMATORS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    bootstrap=True,
)

pipe = Pipeline([
    ("prep", preprocess),
    ("model", rf),
])

# Focused grid after the broad search recorded in docs/model_tuning_history.md.
param_grid = {
    "model__max_features": [0.5, 0.75],
    "model__min_samples_leaf": [1, 2],
    "model__min_samples_split": [2],
    "model__max_depth": [18, None],
}

N_SEARCH_CANDIDATES = len(ParameterGrid(param_grid))
print("Focused tuning candidates:", N_SEARCH_CANDIDATES)
print("Search fits per target:", N_SEARCH_CANDIDATES * SEARCH_CV_FOLDS)
print("Final OOF evaluation fits per target:", EVALUATION_CV_FOLDS)

In [ ]:
metrics = []
prediction_frames = []
importance_frames = []
best_params = {}

total_targets = len(EPVI_TARGETS)
all_started = time.perf_counter()

def fmt_seconds(seconds: float) -> str:
    seconds = int(round(seconds))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"

for target_idx, target in enumerate(EPVI_TARGETS, start=1):
    target_started = time.perf_counter()
    data_t = model_df.dropna(subset=[target]).copy()
    X = data_t[predictor_cols]
    y = data_t[target].astype(float)

    n_search_fits = N_SEARCH_CANDIDATES * search_cv.get_n_splits()
    print("=" * 88, flush=True)
    print(f"[{target_idx}/{total_targets}] Target: {target}", flush=True)
    print(
        f"Rows={len(data_t):,} | predictors={len(predictor_cols):,} | "
        f"focused candidates={N_SEARCH_CANDIDATES} | search folds={search_cv.get_n_splits()} | "
        f"search fits={n_search_fits} | search trees={SEARCH_N_ESTIMATORS}",
        flush=True,
    )

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="r2",
        cv=search_cv,
        n_jobs=-1,
        refit=True,
        verbose=2,
    )

    search_started = time.perf_counter()
    print(f"Starting focused hyperparameter search at {datetime.now().strftime('%H:%M:%S')}...", flush=True)
    search.fit(X, y)
    search_seconds = time.perf_counter() - search_started
    print(
        f"Finished search for {target} in {fmt_seconds(search_seconds)}. "
        f"Best mean search-CV R2={search.best_score_:.4f}",
        flush=True,
    )

    # The focused search uses smaller forests. Evaluation and final outputs use 800 trees.
    best = search.best_estimator_
    best.set_params(model__n_estimators=FINAL_N_ESTIMATORS)
    best_params[target] = {**search.best_params_, "model__n_estimators": FINAL_N_ESTIMATORS}
    print("Best params for final evaluation:", best_params[target], flush=True)

    oof_started = time.perf_counter()
    print(f"Computing out-of-fold predictions ({evaluation_cv.get_n_splits()} fits, {FINAL_N_ESTIMATORS} trees)...", flush=True)
    oof_pred = cross_val_predict(best, X, y, cv=evaluation_cv, n_jobs=-1)
    print(f"OOF predictions finished in {fmt_seconds(time.perf_counter() - oof_started)}.", flush=True)

    refit_started = time.perf_counter()
    print("Refitting best model on all rows...", flush=True)
    best.fit(X, y)
    fitted_pred = best.predict(X)
    print(f"Final refit finished in {fmt_seconds(time.perf_counter() - refit_started)}.", flush=True)

    target_metrics = {
        "target": target,
        "n_rows": len(data_t),
        "n_predictors": len(predictor_cols),
        "best_cv_search_r2_mean": search.best_score_,
        "oof_r2": r2_score(y, oof_pred),
        "oof_mae": mean_absolute_error(y, oof_pred),
        "oof_rmse": rmse(y, oof_pred),
        "oof_spearman": spearman_corr(y, oof_pred),
        "in_sample_r2": r2_score(y, fitted_pred),
        "in_sample_mae": mean_absolute_error(y, fitted_pred),
        "in_sample_rmse": rmse(y, fitted_pred),
    }
    metrics.append(target_metrics)

    prediction_frames.append(pd.DataFrame({
        "ID": data_t["ID"].values,
        "ID_norm": data_t["ID_norm"].values,
        "name": data_t["parish_name"].values,
        "target": target,
        "observed": y.values,
        "predicted_oof": oof_pred,
        "residual_oof": y.values - oof_pred,
        "predicted_in_sample": fitted_pred,
        "residual_in_sample": y.values - fitted_pred,
    }))

    fitted_rf = best.named_steps["model"]
    importances = pd.DataFrame({
        "target": target,
        "feature": predictor_cols,
        "impurity_importance": fitted_rf.feature_importances_,
    }).sort_values("impurity_importance", ascending=False)
    importance_frames.append(importances)

    target_seconds = time.perf_counter() - target_started
    elapsed_total = time.perf_counter() - all_started
    avg_per_target = elapsed_total / target_idx
    remaining = avg_per_target * (total_targets - target_idx)

    print(f"\nMetrics for {target}:", flush=True)
    print(pd.Series(target_metrics).to_string(), flush=True)
    print(
        f"Finished target {target_idx}/{total_targets} in {fmt_seconds(target_seconds)}. "
        f"Elapsed={fmt_seconds(elapsed_total)} | ETA={fmt_seconds(remaining)}",
        flush=True,
    )

metrics_df = pd.DataFrame(metrics).sort_values("oof_r2", ascending=False)
predictions_df = pd.concat(prediction_frames, ignore_index=True)
importances_df = pd.concat(importance_frames, ignore_index=True)

display(metrics_df)

In [ ]:
# Optional, slower but less biased feature importance on the fitted full-data model.
# Set RUN_PERMUTATION_IMPORTANCE = True if you want this diagnostic now.
RUN_PERMUTATION_IMPORTANCE = False

permutation_frames = []
if RUN_PERMUTATION_IMPORTANCE:
    for target in EPVI_TARGETS:
        data_t = model_df.dropna(subset=[target]).copy()
        X = data_t[predictor_cols]
        y = data_t[target].astype(float)

        best = Pipeline([
            ("prep", preprocess),
            ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, bootstrap=True, **{
                k.replace("model__", ""): v for k, v in best_params[target].items()
            })),
        ])
        best.fit(X, y)
        perm = permutation_importance(
            best,
            X,
            y,
            scoring="r2",
            n_repeats=10,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        permutation_frames.append(pd.DataFrame({
            "target": target,
            "feature": predictor_cols,
            "permutation_importance_mean": perm.importances_mean,
            "permutation_importance_std": perm.importances_std,
        }).sort_values("permutation_importance_mean", ascending=False))

permutation_df = pd.concat(permutation_frames, ignore_index=True) if permutation_frames else pd.DataFrame()
if not permutation_df.empty:
    display(permutation_df.groupby("target").head(15))

In [ ]:
# Save modeling outputs outside git.
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
metrics_path = MODEL_OUT_DIR / f"rf_epvi_metrics_{stamp}.csv"
predictions_path = MODEL_OUT_DIR / f"rf_epvi_predictions_residuals_{stamp}.csv"
importances_path = MODEL_OUT_DIR / f"rf_epvi_feature_importances_{stamp}.csv"
params_path = MODEL_OUT_DIR / f"rf_epvi_best_params_{stamp}.json"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
importances_df.to_csv(importances_path, index=False)
with open(params_path, "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False)

if not permutation_df.empty:
    permutation_path = MODEL_OUT_DIR / f"rf_epvi_permutation_importances_{stamp}.csv"
    permutation_df.to_csv(permutation_path, index=False)
else:
    permutation_path = None

print("Wrote:")
for path in [metrics_path, predictions_path, importances_path, params_path, permutation_path]:
    if path is not None:
        print(" -", path)

## Next Step

Use the out-of-fold residuals in `rf_epvi_predictions_residuals_*.csv` as the dependent variables for the residual-explanation stage. That next stage should use the detailed administrative indicators that were intentionally excluded here.